# RetailIQ - Phase 1: Data Ingestion & Schema Exploration

## Project Overview
This notebook covers **Mission 1** of the RetailIQ analytics platform: setting up the raw data ingestion pipeline and loading the multi-table relational retail dataset into a local SQLite database.

### Objectives:
1. **Load Raw Data:** Load the customer, store, product, and sales data from the `data/raw/` folder.
2. **Data Exploration:** Print shape, data types, and the first 5 rows of each table.
3. **Schema Mapping:** Identify and document primary keys (PK) and foreign keys (FK) to establish the star-schema relationships.
4. **Data Quality Assessment:** Flag obvious quality issues (nulls, duplicates, datatype mismatches) without cleaning them.

## 1. Environment Setup & Imports
We import `pandas` for data manipulation, `sqlite3` for local database interaction, and `os` for path management.

In [1]:
import os
import pandas as pd
import sqlite3

# Define paths relative to the project root
RAW_DATA_DIR = os.path.join("..", "data", "raw")
PROCESSED_DATA_DIR = os.path.join("..", "data", "processed")
DB_PATH = os.path.join(PROCESSED_DATA_DIR, "retailiq.db")

print("Setup complete. Paths configured.")

Setup complete. Paths configured.


## 2. Load and Explore Raw Tables
We load each CSV file from `data/raw/` and inspect its shape, columns, datatypes, and first 5 records.

In [2]:
def inspect_table(name, filename):
    path = os.path.join(RAW_DATA_DIR, filename)
    if not os.path.exists(path):
        print(f"File {path} not found!")
        return None
    
    df = pd.read_csv(path)
    print(f"=== Table: {name} ===")
    print(f"Shape: {df.shape}")
    print("\nData Types:")
    print(df.dtypes)
    print("\nFirst 5 Rows:")
    display(df.head())
    print("\nMissing (Null) Values Count:")
    print(df.isnull().sum())
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print("\n" + "="*40 + "\n")
    return df

customers_df = inspect_table("Customers", "customers.csv")
products_df = inspect_table("Products", "products.csv")
stores_df = inspect_table("Stores", "stores.csv")
sales_df = inspect_table("Sales", "sales.csv")

=== Table: Customers ===
Shape: (200, 7)

Data Types:
CustomerID    object
FirstName     object
LastName      object
Gender        object
BirthDate     object
City          object
JoinDate      object
dtype: object

First 5 Rows:


,CustomerID,FirstName,LastName,Gender,BirthDate,City,JoinDate
0,C001,Michael,Davis,M,1996-09-11,Osborneport,2022-09-25
1,C002,Michael,Miller,M,1959-08-18,New Gabrielleport,2020-11-03
2,C003,Carol,Hays,F,2005-04-19,Port Allen,2024-02-12
3,C004,Joseph,Ward,M,1992-06-16,East Edgarborough,2024-09-09
4,C005,Jamie,Salinas,M,1992-06-18,Port Kimberly,2022-02-24



Missing (Null) Values Count:
CustomerID    0
FirstName     0
LastName      0
Gender        0
BirthDate     0
City          0
JoinDate      0
dtype: int64
Duplicate Rows: 0


=== Table: Products ===
Shape: (50, 6)

Data Types:
ProductID       object
ProductName     object
Category        object
SubCategory     object
UnitPrice      float64
CostPrice      float64
dtype: object

First 5 Rows:


,ProductID,ProductName,Category,SubCategory,UnitPrice,CostPrice
0,P001,Like Camera,Electronics,Camera,1673.69,1323.38
1,P002,Audience Television,Electronics,Television,818.76,527.62
2,P003,Here Footwear,Fashion,Footwear,337.63,169.03
3,P004,Four Accessories,Fashion,Accessories,1853.77,1363.52
4,P005,Knowledge Bags,Fashion,Bags,1396.22,1004.09



Missing (Null) Values Count:
ProductID      0
ProductName    0
Category       0
SubCategory    0
UnitPrice      0
CostPrice      0
dtype: int64
Duplicate Rows: 0


=== Table: Stores ===
Shape: (5, 4)

Data Types:
StoreID      object
StoreName    object
City         object
Region       object
dtype: object

First 5 Rows:


,StoreID,StoreName,City,Region
0,S001,MegaMart Jimenezborough,Jimenezborough,South
1,S002,MegaMart Peckmouth,Peckmouth,East
2,S003,MegaMart New Michele,New Michele,West
3,S004,MegaMart Brianahaven,Brianahaven,North
4,S005,MegaMart Johnmouth,Johnmouth,East



Missing (Null) Values Count:
StoreID      0
StoreName    0
City         0
Region       0
dtype: int64
Duplicate Rows: 0


=== Table: Sales ===
Shape: (5000, 8)

Data Types:
TransactionID     object
Date              object
CustomerID        object
ProductID         object
StoreID           object
Quantity           int64
Discount         float64
PaymentMethod     object
dtype: object

First 5 Rows:


,TransactionID,Date,CustomerID,ProductID,StoreID,Quantity,Discount,PaymentMethod
0,T00001,2024-06-18,C160,P014,S003,1,0.10,Bank Transfer
1,T00002,2023-11-02,C171,P030,S004,3,0.15,Bank Transfer
2,T00003,2024-03-28,C142,P002,S002,2,0.15,Mobile Money
3,T00004,2024-06-15,C174,P050,S002,5,0.10,Mobile Money
4,T00005,2024-08-29,C141,P036,S001,3,0.10,Credit Card



Missing (Null) Values Count:
TransactionID    0
Date             0
CustomerID       0
ProductID        0
StoreID          0
Quantity         0
Discount         0
PaymentMethod    0
dtype: int64
Duplicate Rows: 0




## 3. SQLite Database Setup & Ingestion
We ingest each raw dataframe into a local SQLite database at `data/processed/retailiq.db` as-is, mapping dataframes to tables. No cleaning or transformation is performed in this phase.

In [3]:
conn = sqlite3.connect(DB_PATH)

# Ingest tables as-is
print("Loading tables into local SQLite database...")
customers_df.to_sql("customers", conn, if_exists="replace", index=False)
products_df.to_sql("products", conn, if_exists="replace", index=False)
stores_df.to_sql("stores", conn, if_exists="replace", index=False)
sales_df.to_sql("sales", conn, if_exists="replace", index=False)

# Verify tables inside the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in SQLite database:", [t[0] for t in tables])

conn.close()
print("Ingestion complete.")

Loading tables into local SQLite database...


Tables in SQLite database: ['clean_customers', 'clean_products', 'clean_stores', 'clean_sales', 'customers', 'products', 'stores', 'sales']
Ingestion complete.


## 4. Documenting Relational Schema & Integrity

### Primary and Foreign Key Mapping
Based on the table schemas, the logical relationships matching a star-schema model are:
1. **`customers`**:
   - Primary Key: `CustomerID`
2. **`products`**:
   - Primary Key: `ProductID`
3. **`stores`**:
   - Primary Key: `StoreID`
4. **`sales` (Fact Table)**:
   - Primary Key: `TransactionID`
   - Foreign Key: `sales.CustomerID` &rarr; `customers.CustomerID`
   - Foreign Key: `sales.ProductID` &rarr; `products.ProductID`
   - Foreign Key: `sales.StoreID` &rarr; `stores.StoreID`

Let's write code to verify referential integrity (checking for orphan records in the fact table).

In [4]:
conn = sqlite3.connect(DB_PATH)

# Check for orphans in Customers
orphan_cust = pd.read_sql_query(
    "SELECT COUNT(*) FROM sales WHERE CustomerID NOT IN (SELECT CustomerID FROM customers);", 
    conn
).iloc[0, 0]

# Check for orphans in Products
orphan_prod = pd.read_sql_query(
    "SELECT COUNT(*) FROM sales WHERE ProductID NOT IN (SELECT ProductID FROM products);", 
    conn
).iloc[0, 0]

# Check for orphans in Stores
orphan_store = pd.read_sql_query(
    "SELECT COUNT(*) FROM sales WHERE StoreID NOT IN (SELECT StoreID FROM stores);", 
    conn
).iloc[0, 0]

print(f"Orphaned CustomerIDs in Sales: {orphan_cust}")
print(f"Orphaned ProductIDs in Sales: {orphan_prod}")
print(f"Orphaned StoreIDs in Sales: {orphan_store}")

conn.close()

Orphaned CustomerIDs in Sales: 0
Orphaned ProductIDs in Sales: 0
Orphaned StoreIDs in Sales: 0


## 5. Data Quality Observations (Flagged for Mission 2)
Here, we flag obvious data quality issues identified on first glance (without modifying/cleaning the data):

1. **DateTime Fields Stored as Strings/TEXT:**
   - `customers.BirthDate` and `customers.JoinDate` are stored as TEXT rather than DATE/DATETIME.
   - `sales.Date` is stored as TEXT rather than DATE/DATETIME.
2. **Column Casing:**
   - Columns are currently written in `PascalCase`. It is recommended to convert them to standard lowercase `snake_case` (e.g. `customer_id`, `unit_price`) during cleaning for database querying convenience.
3. **Customer City vs Store City:**
   - Both `customers` and `stores` tables have a `City` column. In `sales` queries, this could cause confusion if not properly qualified or renamed (e.g. `customer_city` vs `store_city`).
4. **No Standard Database Constraints:**
   - The loaded tables in SQLite do not have explicit primary key, foreign key, or data type constraints defined (because they were imported as-is via Pandas). In Mission 2, we should create a formal database schema containing these constraints.